# XTTS-v2 Fine-tuning — Клонирование с картавостью

**Что делает этот ноутбук:**
- Принимает 3-10 минут твоего голоса
- Автоматически нарезает и расшифровывает аудио
- Дообучает XTTS-v2 именно на твоём голосе
- Сохраняет картавость и все особенности произношения

**Требования к аудио:**
- Длина: минимум 1 минута (лучше 3-10 минут)
- Формат: WAV или MP3
- Качество: без музыки и эха, чистая речь
- Содержание: обычная речь, побольше слов с буквой Р

**Порядок:**
1. Шаг 1 — проверь GPU (T4)
2. Шаг 2 — установка (5 минут)
3. Шаг 3 — загрузи аудио
4. Шаг 4 — авто-нарезка и расшифровка
5. Шаг 5 — дообучение (30-60 минут)
6. Шаг 6 — тест и скачивание

## Шаг 1: Проверь GPU
Должно написать 'GPU найден'. Если нет — **Runtime → Change runtime type → T4 GPU**

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
if result.returncode == 0:
    print(f'GPU найден: {result.stdout.strip()}')
else:
    print('GPU НЕ найден! Runtime → Change runtime type → T4 GPU')

## Шаг 2: Установка
Занимает 5-7 минут. После завершения — **обязательно Restart session** (Runtime → Restart session).

In [ ]:
# Устанавливаем всё необходимое
!pip install -q coqui-tts
!pip install -q "transformers>=4.46"
!pip install -q openai-whisper
!pip install -q pydub soundfile
!apt-get install -q -y ffmpeg

import os
os.environ["COQUI_TOS_AGREED"] = "1"

print()
print('=' * 50)
print('Установка завершена!')
print('ВАЖНО: сейчас сделай Restart session')
print('Runtime → Restart session')
print('После перезапуска продолжай с Шага 3')
print('=' * 50)

## Шаг 3: Патч совместимости + загрузка аудио
После Restart session начинай отсюда. Шаг 2 больше не нужен.

In [ ]:
import os
os.environ["COQUI_TOS_AGREED"] = "1"

# Патч: восстанавливаем функцию убранную из новых transformers
import torch
import transformers.pytorch_utils as _pt_utils
if not hasattr(_pt_utils, 'isin_mps_friendly'):
    def _isin(e, t): return torch.isin(e, t)
    _pt_utils.isin_mps_friendly = _isin
    print('Патч совместимости применён.')

# Создаём папки
os.makedirs('/content/dataset/wavs', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)
print('Готово. Загружай аудиофайл ниже.')

In [ ]:
from google.colab import files

print('Загрузи свой аудиофайл (WAV или MP3, минимум 1 минута)...')
uploaded = files.upload()

for filename in uploaded.keys():
    audio_path = f'/content/{filename}'
    with open(audio_path, 'wb') as f:
        f.write(uploaded[filename])
    print(f'Загружен: {filename} ({len(uploaded[filename])/1024/1024:.1f} MB)')

print(f'\nПуть к файлу: {audio_path}')

## Шаг 4: Авто-нарезка и расшифровка
Whisper автоматически расшифрует твой голос на русском. Занимает 3-10 минут.

In [ ]:
from pydub import AudioSegment
from pydub.silence import split_on_silence
import soundfile as sf
import numpy as np

print('Загружаем аудио...')
audio = AudioSegment.from_file(audio_path)
audio = audio.set_channels(1).set_frame_rate(22050)  # моно, 22050 Hz

print(f'Длина аудио: {len(audio)/1000:.1f} секунд')
print('Нарезаем на фрагменты...')

# Нарезаем по тишине
chunks = split_on_silence(
    audio,
    min_silence_len=400,
    silence_thresh=audio.dBFS - 16,
    keep_silence=100
)

# Оставляем фрагменты 2-10 секунд
good_chunks = [c for c in chunks if 2000 <= len(c) <= 10000]

# Если мало фрагментов — нарезаем принудительно по 5 сек
if len(good_chunks) < 10:
    print(f'Мало фрагментов ({len(good_chunks)}), нарезаем по 5 секунд...')
    chunk_len = 5000
    good_chunks = [audio[i:i+chunk_len] for i in range(0, len(audio)-chunk_len, chunk_len)]

print(f'Получено фрагментов: {len(good_chunks)}')

# Сохраняем фрагменты
chunk_paths = []
for i, chunk in enumerate(good_chunks):
    path = f'/content/dataset/wavs/chunk_{i:04d}.wav'
    chunk.export(path, format='wav')
    chunk_paths.append(path)

print(f'Сохранено {len(chunk_paths)} аудиофрагментов.')

In [ ]:
import whisper

print('Загружаем Whisper medium (расшифровка займёт несколько минут)...')
whisper_model = whisper.load_model('medium')

transcriptions = []
errors = 0

for i, path in enumerate(chunk_paths):
    try:
        result = whisper_model.transcribe(path, language='ru', fp16=torch.cuda.is_available())
        text = result['text'].strip()
        # Пропускаем пустые и слишком короткие
        if len(text) > 5:
            fname = os.path.basename(path)
            transcriptions.append((fname, text))
            if i % 10 == 0:
                print(f'[{i+1}/{len(chunk_paths)}] {fname}: {text[:60]}...')
    except Exception as e:
        errors += 1

print(f'\nРасшифровано: {len(transcriptions)} фрагментов (ошибок: {errors})')

# Создаём metadata.csv в формате XTTS
metadata_path = '/content/dataset/metadata.csv'
with open(metadata_path, 'w', encoding='utf-8') as f:
    for fname, text in transcriptions:
        # Убираем расширение из имени файла
        name = fname.replace('.wav', '')
        f.write(f'{name}|{text}|speaker\n')

print(f'Датасет создан: {metadata_path}')
print(f'Всего записей: {len(transcriptions)}')

# Показываем первые 5
print('\nПример первых 5 строк:')
with open(metadata_path, encoding='utf-8') as f:
    for line in list(f)[:5]:
        print(line.strip())

## Шаг 5: Fine-tuning XTTS-v2
Дообучение займёт 30-90 минут. Можно оставить работать и вернуться позже.

In [ ]:
# Скачиваем базовую модель XTTS-v2
from TTS.api import TTS
import shutil

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Устройство: {device}')

# Загружаем модель (скачает ~2GB)
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('Базовая модель загружена.')

In [ ]:
from trainer import Trainer, TrainerArgs
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.xtts import Xtts
from TTS.utils.manage import ModelManager

# Находим путь к скачанной модели
manager = ModelManager()
model_path, config_path, _ = manager.download_model('tts_models/multilingual/multi-dataset/xtts_v2')

DATASET_PATH = '/content/dataset'
OUTPUT_PATH = '/content/xtts_finetuned'
LANGUAGE = 'ru'
EPOCHS = 10  # Увеличь до 20-30 для лучшего качества (но дольше)

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Конфигурация обучения
config = XttsConfig()
config.load_json(os.path.join(model_path, 'config.json'))

config.epochs = EPOCHS
config.batch_size = 2
config.eval_batch_size = 2
config.num_loader_workers = 0
config.eval_split_max_size = 256
config.print_step = 50
config.save_step = 1000
config.save_n_checkpoints = 1
config.save_checkpoints = True
config.target_loss = 'loss'
config.print_eval = False
config.run_name = 'xtts_russian_finetune'
config.output_path = OUTPUT_PATH

config.datasets = [{
    'formatter': 'ljspeech',
    'dataset_name': 'my_voice',
    'path': DATASET_PATH,
    'meta_file_train': 'metadata.csv',
    'ignored_speakers': None,
    'language': LANGUAGE,
    'phonemizer': '',
    'meta_file_val': '',
    'ununsed_speakers': None,
}]

print('Конфигурация подготовлена.')
print(f'Эпох обучения: {EPOCHS}')
print(f'Датасет: {DATASET_PATH}')
print(f'Результат: {OUTPUT_PATH}')

In [ ]:
# Инициализируем модель для обучения
model = Xtts.init_from_config(config)
model.load_checkpoint(
    config,
    checkpoint_dir=model_path,
    use_deepspeed=False
)
model.cuda()

# Вычисляем eval_split_size под количество сэмплов
import csv
with open('/content/dataset/metadata.csv', encoding='utf-8') as f:
    n_samples = sum(1 for _ in f)

# Минимум 1 сэмпл на валидацию
eval_split_size = max(0.15, 2.0 / n_samples)
print(f'Сэмплов в датасете: {n_samples}')
print(f'eval_split_size: {eval_split_size:.3f}')

# Загружаем данные
train_samples, eval_samples = load_tts_samples(
    config.datasets,
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=eval_split_size,
)

print(f'Обучающих примеров: {len(train_samples)}')
print(f'Тестовых примеров:  {len(eval_samples)}')

# Запускаем обучение
print('\nЗапускаем fine-tuning... (30-90 минут)')
print('Можно оставить вкладку открытой и вернуться позже.')

trainer = Trainer(
    TrainerArgs(restore_path=None, skip_train_epoch=False),
    config,
    OUTPUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)
trainer.fit()
print('\nFine-tuning завершён!')

## Шаг 6: Тест дообученной модели

In [ ]:
import IPython.display as ipd
import glob

# Находим последний чекпоинт
checkpoints = glob.glob(f'{OUTPUT_PATH}/**/*.pth', recursive=True)
checkpoints = [c for c in checkpoints if 'optimizer' not in c]
latest_checkpoint = sorted(checkpoints)[-1]
print(f'Используем чекпоинт: {latest_checkpoint}')

# Загружаем дообученную модель
config_ft = XttsConfig()
config_ft.load_json(os.path.join(model_path, 'config.json'))

model_ft = Xtts.init_from_config(config_ft)
model_ft.load_checkpoint(config_ft, checkpoint_path=latest_checkpoint, use_deepspeed=False)
model_ft.cuda()

print('Дообученная модель загружена!')

# ==============================
# ВВЕДИ СВОЙ ТЕКСТ ЗДЕСЬ
# ==============================
test_text = "Привет! Это мой голос, дообученный с сохранением картавости. Рыжая лиса прыгнула через реку."

# Используем один из оригинальных фрагментов как эталон
ref_audio = chunk_paths[0]

# Генерируем
out_path = '/content/output/finetuned_result.wav'

gpt_cond_latent, speaker_embedding = model_ft.get_conditioning_latents(
    audio_path=[ref_audio],
    gpt_cond_len=model_ft.config.gpt_cond_len,
    max_ref_length=model_ft.config.max_ref_len,
    sound_norm_refs=model_ft.config.sound_norm_refs,
)

out = model_ft.inference(
    text=test_text,
    language='ru',
    gpt_cond_latent=gpt_cond_latent,
    speaker_embedding=speaker_embedding,
    temperature=0.7,
)

import soundfile as sf
sf.write(out_path, out['wav'], 24000)

print('Результат:')
ipd.display(ipd.Audio(out_path))

## Шаг 7: Скачай результат и модель

In [ ]:
from google.colab import files

# Скачиваем аудио
files.download(out_path)
print('Аудио скачано!')

# Скачиваем чекпоинт модели (чтобы не обучать заново)
print(f'\nЧекпоинт модели: {latest_checkpoint}')
print('Можно скачать его для повторного использования:')
# files.download(latest_checkpoint)  # раскомментируй если нужно